# A5: how many patches per class, and what does each class look like in the spectrum

Two things in one notebook:

1. **Counts.** How many patches does every class have, in EuroSAT (training) and in Thailand (test), and from which study area do the Thai patches come?
   This shows the class imbalance and the "one class, one area" problem that `DATA_NOTES.md` warns about.
2. **Spectral signature.** The median reflectance of every band for one class, drawn as a line (bands from short to long wavelength). It shows what the
   classes look like to the satellite, whether they can be told apart, and where Thailand differs from EuroSAT.

The spectral signatures use the same pixels as A2 to A4 (same seed and order of drawing): 100,000 pixels per class and source,
EuroSAT training patches against the pure Thai patches in L1C and L2A. Grassland is left out of the signatures because no Thai patch passes the purity threshold.

**Keep in mind**
- The Thai images are a January-February median composite; every EuroSAT patch is a single date from any season.
- Each Thai class comes from one or two study areas, so a Thai signature stands for that area, not for the whole country.

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import config as c

# colours on a light surface, from the data-viz palette (each set was checked with its validator)
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SOURCE_COLORS = {"EuroSAT (train)": "#2a78d6", "Thai L1C": "#eb6834", "Thai L2A": "#1baf7a"}   # as in A2 to A4
AREA_COLORS = dict(zip(c.AOIS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))                  # the four study areas
CLASS_COLORS = {"forest": "#008300", "crop": "#e87ba4", "water": "#4a3aa7", "built-up": "#e34948"}   # the four classes

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.size": 10,
})

OUT = c.OUTPUT_DIR / "a5"
OUT.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    # Windows sometimes refuses to open a file that was written a moment ago, so retry a few times
    path = OUT / name
    for attempt in range(5):
        try:
            fig.savefig(path, dpi=130, bbox_inches="tight")
            break
        except OSError:
            if attempt == 4:
                raise
            time.sleep(1)
    print("saved", path)

## 1. Number of patches per class

EuroSAT: the five target classes after merging the ten original ones, split into train, validation and test. Thailand: every patch by the class that covers most of it,
and the usable ("pure": at least 90% one class, no missing pixels) patches by study area.

In [ ]:
EU_Y = np.load(c.EUROSAT_DIR / "eurosat_y.npy")
EU_SPLIT = np.load(c.EUROSAT_DIR / "eurosat_split.npy")             # 0 train, 1 validation, 2 test
TH = pd.read_csv(c.PATCH_DIR / "thailand_patches.csv", keep_default_na=False)   # "nan" is an area name here

eurosat = pd.DataFrame({name: np.bincount(EU_Y[EU_SPLIT == k], minlength=5) for k, name in enumerate(["train", "validation", "test"])},
                       index=c.CLASSES)
eurosat["all"] = eurosat.sum(axis=1)

thai = pd.DataFrame({"all patches (majority class)": TH["label"].value_counts().reindex(c.CLASSES).fillna(0).astype(int),
                     "usable (pure)": TH[TH["pure"]]["label"].value_counts().reindex(c.CLASSES).fillna(0).astype(int)})
by_area = pd.crosstab(TH[TH["pure"]]["label"], TH[TH["pure"]]["area"]).reindex(index=c.CLASSES, columns=list(c.AOIS), fill_value=0)

counts = pd.concat({"EuroSAT": eurosat, "Thailand": thai, "Thailand, usable, by area": by_area}, axis=1)
counts.to_csv(OUT / "patch_counts.csv")
counts

In [ ]:
usable = thai["usable (pure)"][c.THAI_EVAL_CLASSES]
train = eurosat["train"]
print(f"EuroSAT train: largest class {train.idxmax()} {train.max()}, smallest {train.idxmin()} {train.min()}, ratio {train.max() / train.min():.1f}")
print(f"Thai usable patches (4 evaluated classes): largest {usable.idxmax()} {usable.max()}, smallest {usable.idxmin()} {usable.min()}, ratio {usable.max() / usable.min():.1f}")

### Figure: patches per class

Left: EuroSAT training patches. Right: usable Thai patches, split by study area (each class comes almost entirely from one area; grassland has none).

In [ ]:
def bar_labels(ax, totals):
    for x, total in enumerate(totals):
        ax.text(x, total, f"{int(total):,}", ha="center", va="bottom", fontsize=10, color=INK)


fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
x = np.arange(len(c.CLASSES))

left.bar(x, eurosat["train"], width=0.62, color=MUTED)
bar_labels(left, eurosat["train"])
left.set_title("EuroSAT: training patches", loc="left", fontsize=12, color=INK)

bottom = np.zeros(len(c.CLASSES))
for area in c.AOIS:
    values = by_area[area].to_numpy()
    right.bar(x, values, width=0.62, bottom=bottom, color=AREA_COLORS[area], edgecolor=SURFACE, linewidth=1.5, label=area)
    bottom += values
bar_labels(right, bottom)
right.set_title("Thailand: usable (pure) patches, by study area", loc="left", fontsize=12, color=INK)
right.legend(handles=[Patch(color=AREA_COLORS[a], label=a) for a in c.AOIS], frameon=False, loc="upper right", title="study area")

for ax in (left, right):
    ax.set_xticks(x)
    ax.set_xticklabels(c.CLASSES)
left.set_ylabel("number of patches")
fig.tight_layout()
save(fig, "patches_per_class.png")
plt.show()

## 2. Spectral signature of every class

The same pixels as A2 to A4. For every class, source and band the median reflectance (x 10000) is computed; the 25th and 75th percentiles are kept in the table.

In [ ]:
EU_X = np.load(c.EUROSAT_DIR / "eurosat_X.npy", mmap_mode="r")
TH_X = {"Thai L1C": np.load(c.PATCH_DIR / "thailand_X_L1C.npy", mmap_mode="r"),
        "Thai L2A": np.load(c.PATCH_DIR / "thailand_X_L2A.npy", mmap_mode="r")}
EVAL = c.THAI_EVAL_CLASSES      # forest, crop, built-up, water
PATCHES_PER_CLASS = 300
PIXELS_PER_CLASS = 100_000


def draw(patch_ids, rng):
    ids = np.sort(rng.choice(patch_ids, size=min(PATCHES_PER_CLASS, len(patch_ids)), replace=False))
    pix = np.sort(rng.choice(len(ids) * 64 * 64, size=min(PIXELS_PER_CLASS, len(ids) * 64 * 64), replace=False))
    return ids, pix


def take(X, ids, pix):
    block = np.asarray(X[ids])                                  # (patches, 12, 64, 64)
    return block.transpose(0, 2, 3, 1).reshape(-1, 12)[pix].astype("float32")


rng = np.random.default_rng(c.SEED)              # same order of drawing as in A2 to A4
signature = {name: {} for name in SOURCE_COLORS}
for cls in EVAL:
    eu_ids = np.flatnonzero((EU_Y == c.CLASSES.index(cls)) & (EU_SPLIT == 0))
    ids, pix = draw(eu_ids, rng)
    signature["EuroSAT (train)"][cls] = take(EU_X, ids, pix)
    th_ids = TH.loc[TH["pure"] & (TH["label"] == cls), "patch_id"].to_numpy()
    ids, pix = draw(th_ids, rng)
    for name, X in TH_X.items():
        signature[name][cls] = take(X, ids, pix)

records = []
for name, per_class in signature.items():
    for cls, px in per_class.items():
        p25, med, p75 = np.percentile(px, (25, 50, 75), axis=0)
        for b, band in enumerate(c.BANDS):
            records.append({"source": name, "class": cls, "band": band, "p25": p25[b], "median": med[b], "p75": p75[b]})
sig = pd.DataFrame(records)
sig.round(0).to_csv(OUT / "signatures.csv", index=False)
print(len(sig), "rows")

### Figure: one class at a time, Europe against Thailand

Each panel is one class. The horizontal axis lists the bands from short to long wavelength (equal spacing, not to scale):
blue to red (B1 to B4), the red-edge and near-infrared bands (B5 to B9), then the short-wave infrared (B11, B12). Lines that coincide mean the two regions look alike in the spectrum.

In [ ]:
x = np.arange(len(c.BANDS))
fig, axes = plt.subplots(2, 2, figsize=(13, 8.4), sharex=True)
for ax, cls in zip(axes.ravel(), EVAL):
    for name, col in SOURCE_COLORS.items():
        part = sig[(sig["source"] == name) & (sig["class"] == cls)].set_index("band").reindex(c.BANDS)
        ax.plot(x, part["median"], color=col, linewidth=2, marker="o", markersize=5, label=name)
    ax.set_title(cls, loc="left", fontsize=12, color=INK)
    ax.set_xticks(x)
    ax.set_xticklabels(c.BANDS)
    ax.grid(axis="x", visible=False)
for ax in axes[:, 0]:
    ax.set_ylabel("median reflectance x 10000")
handles = [Line2D([0], [0], color=col, linewidth=2.5, marker="o", markersize=5, label=name) for name, col in SOURCE_COLORS.items()]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Spectral signature of every class: EuroSAT against Thailand", y=1.04, fontsize=13, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, "signatures_by_class.png")
plt.show()

### Figure: the four classes side by side, inside one region

If two classes have nearly the same line in a region, they are hard to tell apart there. Compare the three panels: does a pair of classes that is easy in EuroSAT come closer in Thailand?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for ax, name in zip(axes, SOURCE_COLORS):
    for cls in ["forest", "crop", "water", "built-up"]:
        part = sig[(sig["source"] == name) & (sig["class"] == cls)].set_index("band").reindex(c.BANDS)
        ax.plot(x, part["median"], color=CLASS_COLORS[cls], linewidth=2, marker="o", markersize=5, label=cls)
    ax.set_title(name, loc="left", fontsize=12, color=INK)
    ax.set_xticks(x)
    ax.set_xticklabels(c.BANDS, rotation=45)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("median reflectance x 10000")
handles = [Line2D([0], [0], color=CLASS_COLORS[cls], linewidth=2.5, marker="o", markersize=5, label=cls) for cls in ["forest", "crop", "water", "built-up"]]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Spectral signature of the classes inside one region", y=1.07, fontsize=13, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
save(fig, "signatures_by_region.png")
plt.show()

## 3. The numbers behind the signatures

Median reflectance (x 10000) of every band, per class, for the three sources (the table view of the figures above):

In [ ]:
table = sig.pivot_table(index=["class", "band"], columns="source", values="median").reindex(
    pd.MultiIndex.from_product([EVAL, c.BANDS], names=["class", "band"]))
table[["EuroSAT (train)", "Thai L1C", "Thai L2A"]].round(0).astype(int)

## 4. Questions to answer for the slide

- How unbalanced are the two sides, and in which direction? (EuroSAT: most built-up; Thailand: most forest.)
- Which Thai classes come from a single area? What does that mean for what a Thai result can claim?
- In which bands do the classes separate most, and where do they come close (inside one region)?
- Does a pair of classes that is easy to tell apart in EuroSAT come closer together in Thailand?